# BERTopic — Tweede Kamer corpus

Adapted from `BERT/03_BERT.ipynb` (same Dutch-BERT-embedding + UMAP + HDBSCAN +
BERTopic pipeline used for news/talkshows) for the tweede kamer documents.

**Differences from the news version:**
- `tweede_kamer_data.csv` has no `processed`/`nouns`/`adjectives`/`verbs` columns yet,
  so this notebook adds the same spaCy lemmatisation step that produced them for news
  (originally in `preprocessing/preprocessing.ipynb`).
- Sub-unit column is `type` (plenair verslag / beleidsnota / vergaderstuk), not `outlet`.
- Corpus is far smaller (~844 docs vs. ~15–16k for news) — UMAP/HDBSCAN params are
  carried over unchanged from the news notebook as a starting point, but may need
  retuning (e.g. `min_cluster_size`) if the outlier rate or topic count looks off.
- `custom_stopwords` is intentionally empty to start — the news list was tuned for
  newspaper-scrape artifacts (LexisNexis boilerplate etc.) that don't apply here.
  Add kamer-specific junk tokens after inspecting the first topic run.

In [ ]:
import pathlib

NB_DIR = pathlib.Path(".").resolve()

# ============================================================
# PARAMETERS — edit before running
# ============================================================

INPUT_CSV   = "../tweede_kamer_data.csv"
SUBUNIT_COL = "type"   # plenair verslag / beleidsnota / vergaderstuk

RANDOM_SEED = 205   # used for UMAP random_state; carried over from the news run

CUSTOM_STOPWORDS = [
    # add kamer-specific junk tokens here after inspecting topic output,
    # e.g. PDF page-number artifacts, document boilerplate
]

EMBED_MODEL = "GroNLP/bert-base-dutch-cased"

OUT_CSV         = f"tk_bert_{RANDOM_SEED}.csv"
OUT_MODEL       = f"bertopic_model_tk_{RANDOM_SEED}"
OUT_TOPICS_XLSX = f"topics_overview_tk_{RANDOM_SEED}.xlsx"
# ============================================================

In [ ]:
# imports
import re
from collections import Counter

import numpy as np
import pandas as pd
import spacy
import torch
from bertopic import BERTopic
from gensim.models import Phrases
from gensim.models.phrases import Phraser
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from transformers import AutoModel, AutoTokenizer
from umap import UMAP

## 1. Load & filter

In [ ]:
df = pd.read_csv((NB_DIR / INPUT_CSV).resolve())
before = len(df)
df = df[df["body"].notna()].reset_index(drop=True)
print(f"Loaded {before} rows, {before - len(df)} dropped for empty body, {len(df)} remain")

print()
print(f"Sub-unit breakdown ({SUBUNIT_COL}):")
print(df[SUBUNIT_COL].value_counts().to_string())

## 2. Preprocessing — build `processed` column

Same lemmatisation logic used to build `processed`/`nouns`/`adjectives`/`verbs` for
news (`preprocessing/preprocessing.ipynb`): strip URLs/mentions, lowercase, then
spaCy `nl_core_news_lg` lemmas — nouns filtered to non-stop/non-punct, concatenated
with all adjective and verb lemmas (order: nouns + adjectives + verbs).

In [ ]:
nlp_pre = spacy.load("nl_core_news_lg")


def preprocess(text):
    text = re.sub(r"http\S+|www\S+|https\S+", "", text, flags=re.MULTILINE)
    text = re.sub(r"\@\w+|\#", "", text)
    text = text.lower().strip()

    doc = nlp_pre(text)

    nouns = [
        token.lemma_ for token in doc
        if not token.is_stop
        and not token.is_punct
        and token.pos_ == "NOUN"
        and len(token.lemma_) > 1
    ]
    adjectives = [token.lemma_ for token in doc if token.pos_ == "ADJ"]
    verbs      = [token.lemma_ for token in doc if token.pos_ == "VERB"]

    processed = " ".join(nouns + adjectives + verbs)
    return processed, " ".join(nouns), " ".join(adjectives), " ".join(verbs)


results = df["body"].astype(str).apply(preprocess)
df[["processed", "nouns", "adjectives", "verbs"]] = pd.DataFrame(
    results.tolist(), index=df.index
)

print("Preprocessing complete.")
df[["title", "processed"]].head(3)

## 3. Bigrams + custom stop-word removal

In [ ]:
def remove_words(text, words_to_remove, min_length=3):
    if not words_to_remove:
        return " ".join(tok for tok in text.split() if len(tok) >= min_length)
    escaped = [re.escape(w) for w in words_to_remove]
    pat = r"(?<!\S)(?:" + "|".join(escaped) + r")(?!\S)"
    cleaned = re.sub(pat, "", text, flags=re.IGNORECASE)
    cleaned = " ".join(cleaned.split())
    return " ".join(tok for tok in cleaned.split() if len(tok) >= min_length)


sentences = [doc.split() for doc in df["processed"]]
bigram    = Phrases(sentences, min_count=10, threshold=150)
bigram_ph = Phraser(bigram)

df["processed"] = (
    df["processed"]
    .apply(lambda txt: " ".join(bigram_ph[txt.split()]))
    .apply(lambda txt: remove_words(txt, CUSTOM_STOPWORDS, min_length=3))
)
df = df[df["processed"].str.strip() != ""].reset_index(drop=True)
texts = df["processed"].tolist()

print(f"{len(texts)} documents remain after bigram/stopword pass")

## 4. Dutch stop-words & vectorizer

In [ ]:
dutch_stops = list(nlp_pre.Defaults.stop_words)
vectorizer = CountVectorizer(
    stop_words=dutch_stops,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.6,
)

## 5. UMAP & HDBSCAN

Params carried over unchanged from the news pipeline (tuned for ~15–16k docs).
With only ~844 kamer documents, watch the outlier rate and topic count after the
first run — `min_cluster_size`/`min_samples` may need lowering if too much gets
marked as noise, or raising if topics fragment too finely.

In [ ]:
umap_model = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.1,
    random_state=RANDOM_SEED,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=5,
    min_samples=8,
    cluster_selection_epsilon=0.1,
    cluster_selection_method="eom",
    metric="euclidean",
    prediction_data=True,
)

## 6. Dutch BERT embeddings

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)
hf_model  = AutoModel.from_pretrained(EMBED_MODEL)
hf_model.eval()


def embed_texts(texts, batch_size=32):
    embs = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
            out = hf_model(**enc).last_hidden_state
            mask = enc.attention_mask.unsqueeze(-1).float()
            summed = (out * mask).sum(1)
            counts = mask.sum(1).clamp(min=1e-9)
            embs.append((summed / counts).cpu().numpy())
    return np.vstack(embs)


embeddings = embed_texts(texts, batch_size=16)
print(f"Embeddings: {embeddings.shape}")

In [ ]:
class HFEmbedder:
    def embed_documents(self, docs):
        return embed_texts(docs)

    def embed_queries(self, docs):
        return embed_texts(docs)

## 7. Fit BERTopic

In [ ]:
topic_model = BERTopic(
    embedding_model=HFEmbedder(),
    vectorizer_model=vectorizer,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True,
)

topics, probs = topic_model.fit_transform(texts)

initial_noise = (np.array(topics) == -1).mean()
print(f"Initial topics: {len(set(topics)) - (1 if -1 in topics else 0)}, "
      f"outlier rate: {initial_noise:.2%}")

## 8. Reduce outliers & finalise topic representations

In [ ]:
new_topics = topic_model.reduce_outliers(
    documents=texts,
    topics=topics,
    threshold=0.25,
    strategy="embeddings",
)

topic_model.update_topics(texts, new_topics)
topic_model.topics_    = new_topics
topic_model.documents_ = texts
topic_model.topic_sizes_ = dict(Counter(new_topics))

topic_info = topic_model.get_topic_info()

NEW_TOP_N_WORDS = 30
reprs = []
for tid in topic_info.Topic:
    if tid == -1:
        reprs.append("Outlier Topic / Not Applicable")
    else:
        words_scores = topic_model.get_topic(tid)
        reprs.append(", ".join(w for w, _ in words_scores[:NEW_TOP_N_WORDS]))
topic_info[f"Representation_Top_{NEW_TOP_N_WORDS}"] = reprs

with pd.option_context("display.max_colwidth", None):
    print(topic_info[["Topic", "Count", "Name", f"Representation_Top_{NEW_TOP_N_WORDS}"]].head(50))

## 9. Assign topics back to the dataframe & sanity-check

In [ ]:
df["topic"]       = new_topics
df["probability"] = [probs[i, t] if t >= 0 else 0.0 for i, t in enumerate(new_topics)]

final_noise = (df["topic"] == -1).mean()
print(f"Final noise rate: {final_noise:.2%}")
print(f"Final topic count (excl. outlier): {df.loc[df['topic'] != -1, 'topic'].nunique()}")

assert len(new_topics) == len(df)
assert probs.shape[0] == len(df)
for idx in [0, len(df)//2, len(df)-1]:
    assigned_t = df.at[idx, "topic"]
    assigned_p = df.at[idx, "probability"]
    recomputed_p = probs[idx, assigned_t] if assigned_t >= 0 else 0.0
    assert abs(assigned_p - recomputed_p) < 1e-8
print("Lengths and spot-checks line up.")

## 10. Export

In [ ]:
out_csv_path   = NB_DIR / OUT_CSV
out_model_path = NB_DIR / OUT_MODEL
out_xlsx_path  = NB_DIR / OUT_TOPICS_XLSX

df.to_csv(out_csv_path)
topic_model.save(str(out_model_path))
topic_info.to_excel(out_xlsx_path, index=False)

print(f"Saved dataframe      -> {out_csv_path.name}")
print(f"Saved BERTopic model -> {out_model_path.name}")
print(f"Saved topics summary -> {out_xlsx_path.name}")